# Nexora Native v1 — Kaggle Pretraining

**架構：** Decoder-only Transformer，6.902M 參數，從零隨機初始化  
**完全不依賴任何外部 pretrained weights（GPT-2 / CKIP / UER）**

使用 NEXUX 自有語料（`data/*.jsonl`），自有 char-level tokenizer（nexora-char-v1）

In [ ]:
# 確認 GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU detected')

In [ ]:
# clone repo（depth=1 跳過大型舊 commit，只取最新狀態）
!git clone --depth 1 --branch claude/hello-682f43 https://github.com/liug1217/NEXUX.git /kaggle/working/NEXUX
%cd /kaggle/working/NEXUX
!ls -la nexora/

In [ ]:
# 確認 data/ 語料檔案
import glob
jsonl_files = sorted(glob.glob('data/*.jsonl'))
print(f'語料檔案數：{len(jsonl_files)}')
print('前 5 個：', jsonl_files[:5])

In [ ]:
# 確認 PyTorch 版本與 CUDA
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 正式訓練 5000 步
# - 從零初始化，不載入任何外部 pretrained weights
# - CUDA + AMP 自動啟用
# - 每 500 步評估 val loss，每 1000 步存 checkpoint
# - 保留最佳 val checkpoint（nexora/checkpoints/nexora_best.pt）

!python -m nexora.train --steps 5000

In [ ]:
# 訓練完成後評估
!python -m nexora.eval

In [ ]:
# 列出 checkpoint 檔案
import os
ck_dir = 'nexora/checkpoints'
if os.path.exists(ck_dir):
    for f in os.listdir(ck_dir):
        path = os.path.join(ck_dir, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f'{f}  {size_mb:.1f} MB')
else:
    print('checkpoints 目錄不存在')

In [ ]:
# 下載 best checkpoint（Kaggle output 路徑）
import shutil, os

src_best = 'nexora/checkpoints/nexora_best.pt'
src_latest = 'nexora/checkpoints/nexora_latest.pt'
dst_dir = '/kaggle/working'

for src in [src_best, src_latest]:
    if os.path.exists(src):
        dst = os.path.join(dst_dir, os.path.basename(src))
        shutil.copy2(src, dst)
        print(f'已複製 {src} → {dst}  ({os.path.getsize(dst)/1e6:.1f} MB)')
    else:
        print(f'找不到 {src}')